# QUA³CK — Q-Phase: Frage & Fragestellung

**Projekt:** Unfallatlas Deutschland · ML-Portfolio  
**Autor:** Jonas Weirauch  
**Datum:** Mai 2026

---

## Übersicht

In der **Q-Phase** des QUA³CK-Frameworks werden Forschungsfrage, Hypothesen und Erfolgsmetriken festgelegt — bevor ein einziges Modell trainiert wird. Dies entspricht dem wissenschaftlichen Prinzip der *pre-registration* und verhindert, dass die Fragestellung im Nachhinein an die Ergebnisse angepasst wird.

| QUA³CK-Phase | Notebook | Status |
|---|---|---|
| Q — Question | `01_Q_Phase.ipynb` | ✅ |
| U — Understanding | `02_U_Phase.ipynb` | 🔄 |
| A³ — Algorithm/Adapt/Adjust | `03_A3_Phase.ipynb` | 🔄 |
| C — Conclude & Compare | `04_C_Phase.ipynb` | 🔄 |
| K — Knowledge Transfer | `app/streamlit_app.py` | 🔄 |

## 0 — Setup & Datei-Überblick

In [ ]:
from pathlib import Path
import duckdb
import pandas as pd

DATA = Path("..") / "data" / "body.parquet"
assert DATA.exists(), f"Datei nicht gefunden: {DATA.resolve()}"

con = duckdb.connect()
print(f"DuckDB {duckdb.__version__} | Pandas {pd.__version__}")

In [ ]:
schema = con.execute(f"DESCRIBE SELECT * FROM '{DATA}'").df()
print(f"Spalten: {len(schema)}")
schema

In [ ]:
shape = con.execute(f"SELECT COUNT(*) as zeilen, COUNT(DISTINCT UJAHR) as jahre FROM '{DATA}'").df()
print(shape.to_string(index=False))

years = con.execute(f"""
    SELECT UJAHR, COUNT(*) as unfaelle
    FROM '{DATA}'
    GROUP BY UJAHR
    ORDER BY UJAHR
""").df()
print("\nUnfälle pro Jahr:")
print(years.to_string(index=False))

In [ ]:
cat = con.execute(f"""
    SELECT
        UKATGEORIE,
        CASE UKATGEORIE WHEN 1 THEN 'Getötet' WHEN 2 THEN 'Schwer verletzt' ELSE 'Leicht verletzt' END AS Schweregrad,
        COUNT(*) AS n,
        ROUND(100.0 * COUNT(*) / SUM(COUNT(*)) OVER (), 1) AS pct
    FROM '{DATA}'
    GROUP BY UKATGEORIE
    ORDER BY UKATGEORIE
""").df()
print("Zielvariable UKATGEORIE:")
cat

In [ ]:
con.execute(f"SELECT * FROM '{DATA}' LIMIT 5").df()

## 1 — Forschungsfrage

> **Welche raumzeitlichen, infrastrukturellen und meteorologischen Faktoren bestimmen die Schwere eines Verkehrsunfalls in Deutschland, und lässt sich diese Schwere mit interpretierbaren Machine-Learning-Modellen aus öffentlich verfügbaren Daten zuverlässig vorhersagen?**

### Motivation

Straßenverkehrsunfälle sind eine der führenden Todesursachen weltweit. Die EU hat mit **Vision Zero 2050** das Ziel formuliert, Verkehrstote auf null zu reduzieren. Deutschland verzeichnet laut Statistischem Bundesamt jährlich etwa 2.700–3.200 Verkehrstote und über 300.000 Verletzte (Stand 2024).

Der Unfallatlas des Statistischen Bundesamts ist die einzige öffentlich verfügbare, georeferenzierte Vollerhebung polizeilich aufgenommener Personenschadensunfälle in Deutschland. Mit **2,09 Millionen Einträgen über 9 Jahre (2016–2024)** bietet er eine ideale Basis für datengetriebene Unfallschwere-Analyse.

### Warum diese Frage?

| Kriterium | Erfüllt? |
|---|---|
| Klares ML-Target (Klassifikation) | ✅ `UKATGEORIE` 1/2/3 |
| Wissenschaftlich anschlussfähig | ✅ Peer-reviewed Literatur 2022–2025 nutzt XGBoost + SHAP auf ähnlichen Daten |
| Gesellschaftliche Relevanz | ✅ EU Vision Zero 2050 |
| Regional fokussierbar | ✅ Hessen/Wiesbaden-Zoom möglich |
| Nicht trivial zu schlagen | ✅ Klassenimbalance 1%/18%/81% macht Majority-Class-Baseline wertlos |

## 2 — Datensatz-Beschreibung

### Quelle

- **Plattform:** [GovData.de](https://www.govdata.de/suche/daten/unfallatlas) — Deutschlands nationales Open-Data-Portal, indexiert auf data.europa.eu
- **Bereitsteller:** Mobilithek / Statistisches Bundesamt
- **Lizenz:** Datenlizenz Deutschland – Namensnennung – Version 2.0 (entspricht CC-BY)
- **Zeitraum:** 2016–2024 (9 Jahrgänge)
- **Format im Projekt:** `data/body.parquet` — 2.092.401 Zeilen, 21 Spalten

### Schema

| Spalte | Typ | Bedeutung | Kodierung |
|---|---|---|---|
| `OBJECTID` | INTEGER | Eindeutige Unfall-ID | — |
| `UJAHR` | SMALLINT | Unfalljahr | 2016–2024 |
| `UMONAT` | TINYINT | Unfallmonat | 1–12 |
| `USTUNDE` | TINYINT | Unfallstunde | 0–23 |
| `UWOCHENTAG` | TINYINT | Wochentag | 1=So, 2=Mo, …, 7=Sa |
| **`UKATGEORIE`** | TINYINT | **Zielvariable: Unfallschwere** | **1=Getötet, 2=Schwer, 3=Leicht** |
| `UART` | TINYINT | Unfallart | 0–9 (10 Klassen) |
| `UTYP1` | TINYINT | Unfalltyp | 1–7 |
| `ULICHTVERH` | TINYINT | Lichtverhältnisse | 0=Tageslicht, 1=Dämmerung, 2=Dunkelheit |
| `STRZUSTAND` | TINYINT | Straßenzustand | 0=trocken, 1=nass/feucht, 2=winterglatt |
| `IstRad` | BOOLEAN | Fahrradbeteiligung | True/False |
| `IstPKW` | BOOLEAN | PKW-Beteiligung | True/False |
| `IstFuss` | BOOLEAN | Fußgängerbeteiligung | True/False |
| `IstKrad` | BOOLEAN | Krad/Motorrad | True/False |
| `IstGkfz` | BOOLEAN | Güterkraftfahrzeug | True/False |
| `IstSonstig` | BOOLEAN | Sonstiges Verkehrsmittel | True/False |
| `LON` | DOUBLE | Längengrad WGS84 | Dezimalgrad |
| `LAT` | DOUBLE | Breitengrad WGS84 | Dezimalgrad |
| `UREGBEZ` | VARCHAR | Regierungsbezirk-Code | — |
| `UKREIS` | VARCHAR | Kreis-Code (5-stellig) | Erste 2 Ziffern = Bundesland |
| `UGEMEINDE` | VARCHAR | Gemeinden-Code | — |

> **Hinweis:** Das Feld heißt im Datensatz `UKATGEORIE` (Tippfehler, fehlendes K) — nicht `UKATEGORIE` wie in der offiziellen Dokumentation.

### Bekannte Limitationen (DIG-Introspektion)

| Limitation | Implikation |
|---|---|
| Nur Personenschadensunfälle | Sachschadensunfälle (~70% aller Unfälle) fehlen vollständig — **Selektionsbias** |
| 92%-Geocoding-Quote | ~8% der Unfälle werden nicht veröffentlicht (nicht eindeutig geocodierbar) — **Selektionsbias** |
| Keine Demografie | Alter und Geschlecht der Beteiligten fehlen — laut Literatur starke Prädiktoren |
| Keine Geschwindigkeit | Nur erlaubte Geschwindigkeit via OSM annäherbar, keine Fahrzeugdaten |
| Keine Unfallursache | Nur Kategorien, kein Freitext |
| Keine Nicht-Meldungen | Dunkelziffer nicht beobachtbar |

## 3 — Hypothesen

Die folgenden Hypothesen sind aus der Literatur abgeleitet und werden in der A³-Phase empirisch überprüft. Sie sind **vor der Modellierung** formuliert.

| Nr. | Hypothese | Feature | Erwartete Richtung | Literatur |
|---|---|---|---|---|
| H1 | Unfälle bei Dunkelheit sind schwerer als bei Tageslicht | `ULICHTVERH` | Dunkelheit → höhere Schwere | Petzoldt et al. 2023 |
| H2 | Winterglatte Straßen erhöhen die Unfallschwere | `STRZUSTAND` | Glatteis > nass > trocken | Theofilatos & Yannis 2014 |
| H3 | Nacht-/Wochenendunfälle sind schwerer (kombinierter Effekt) | `USTUNDE × UWOCHENTAG` | Freitagabend–Sonntag & 0–6 Uhr → höhere Schwere | Schlößler et al. 2024 |
| H4 | Fahrrad- und Fußgängerbeteiligung erhöht die Schwere | `IstRad`, `IstFuss` | Ungeschützte Verkehrsteilnehmer → höhere Schwere | Santos et al. 2022 |
| H5 | Ländliche Kreise haben schwerere Unfälle als städtische Kreise | `UKREIS` (→ ULAND) | Ländliche Gebiete → höhere Schwere (Geschwindigkeit, Rettungszeiten) | DESTATIS 2024 |
| H6 | Unfallart und Unfalltyp sind die stärksten Prädiktoren | `UART`, `UTYP1` | SHAP-Beiträge > zeitliche Features | Pakgohar et al. 2021 |
| H7 | Die Stundenverteilung zeigt bimodales Muster mit Pendler- und Freizeitspitzen | `USTUNDE` | Peaks um 7–9 Uhr und 15–18 Uhr | BASt 2023 |

### Überprüfungsplan

- **H1–H2:** Univariate Analyse (Cramér's V, bedingte Mittelwert-Tabellen) in `02_U_Phase.ipynb`
- **H3:** Heatmap Wochentag × Stunde × mittlere Schwere in `02_U_Phase.ipynb`
- **H4–H5:** Feature Importance + SHAP in `04_C_Phase.ipynb`
- **H6:** SHAP Summary Plot — Top-Features aus bestem Modell in `04_C_Phase.ipynb`
- **H7:** Stundenprofil-Plot in `02_U_Phase.ipynb`

## 4 — Erfolgsmetriken

### Primärmetrik

**macro-F1** auf dem Held-Out-Testset (2024) — gewichtet alle drei Klassen gleich, damit Klasse 1 (Getötete, 1% der Daten) nicht ignoriert wird.

### Zielwerte

| Modell / Baseline | Erwarteter macro-F1 | Recall Klasse 1 (Getötet) |
|---|---|---|
| Random Guess | ~0.33 | ~0.33 |
| Majority Class (immer Klasse 3) | ~0.30 | 0.00 |
| Logistic Regression | 0.42–0.48 | 0.10–0.20 |
| Random Forest | 0.50–0.55 | 0.25–0.35 |
| XGBoost (default) | 0.55–0.60 | 0.30–0.40 |
| LightGBM + Class Weights | 0.60–0.65 | 0.45–0.55 |
| **CatBoost + Threshold Moving** | **0.65–0.72** | **0.55–0.70** |

**Projektziel:** macro-F1 ≥ 0.55 (Primär), Recall Klasse 1 ≥ 0.50 (Sekundär).

### Test-Split-Strategie

```
Train: 2016–2022   (~1.55 Mio. Zeilen, ~74%)
Val:   2023        (~269 Tsd. Zeilen, ~13%)
Test:  2024        (~269 Tsd. Zeilen, ~13%)
```

**Warum chronologisch?** Zufälliger Split würde Data Leakage erzeugen und die Generalisierung überschätzen. Das Testjahr 2024 war zur Trainingszeit vollständig unbekannt — das entspricht echtem Deployment-Szenario.

### Zusatzmetriken

- Konfusionsmatrix (normalisiert) für alle Modelle
- ROC-AUC (one-vs-rest) pro Klasse
- Precision-Recall-Kurven (besonders wichtig wegen Imbalance)
- SHAP-Feature-Importance für bestes Modell

In [ ]:
split = con.execute(f"""
    SELECT
        CASE
            WHEN UJAHR <= 2022 THEN 'Train (2016–2022)'
            WHEN UJAHR = 2023  THEN 'Val   (2023)'
            ELSE                    'Test  (2024)'
        END AS split,
        COUNT(*) AS n,
        ROUND(100.0 * COUNT(*) / SUM(COUNT(*)) OVER (), 1) AS pct
    FROM '{DATA}'
    GROUP BY split
    ORDER BY MIN(UJAHR)
""").df()
split

## 5 — Literatur

| Quelle | Methodik | Relevanz |
|---|---|---|
| **Santos et al. (2022)**, *Accident Analysis & Prevention* | XGBoost + SHAP auf portugiesischen Unfalldaten | Baseline-Methodik; SHAP zeigt Fahrrad/Fußgänger als Hauptfaktor |
| **Pakgohar et al. (2021)**, *IATSS Research* | LightGBM + SMOTE | Imbalance-Behandlung; LightGBM > Random Forest |
| **Schlößler et al. (2024)**, *Accident Analysis & Prevention* | ML-Ensemble auf deutschen Unfalldaten | Direkt vergleichbar — Deutsche Unfallstatistik |
| **MDPI Sustainability (2024)** | CatBoost + Threshold Moving | Threshold Moving für seltene Klassen |
| **BASt (2023)**, *Unfallentwicklung auf deutschen Straßen* | Deskriptive Statistik | Offizielle Stunden-/Wochentagsmuster als Ground Truth |

### Wichtigste Erkenntnisse

1. **LightGBM und CatBoost** dominieren bei tabellarischen Verkehrsunfalldaten (macro-F1 typisch 0.60–0.72).
2. **SHAP** ist der Standard für Erklärbarkeit bei Boosting-Modellen.
3. **Threshold Moving** ist oft effektiver als SMOTE für stark imbalancierte Daten.
4. **Ordinale Klassifikation** (1 < 2 < 3) kann macro-F1 um ~2–4% verbessern.
5. **Fehlende Demografie** ist die häufigste Limitation in Studien mit öffentlichen Daten — muss explizit diskutiert werden.

## 6 — Zusammenfassung Q-Phase

| Aspekt | Festlegung |
|---|---|
| **Forschungsfrage** | Schweregrad-Klassifikation (1/2/3) aus raumzeitlichen + infrastrukturellen Features |
| **Datensatz** | Unfallatlas 2016–2024, 2.09 Mio. Unfälle, GovData (Datenlizenz Deutschland 2.0) |
| **Zielvariable** | `UKATGEORIE`: 1=Getötet (1%), 2=Schwer (18%), 3=Leicht (81%) |
| **Primärmetrik** | macro-F1 ≥ 0.55 auf Held-Out 2024 |
| **Sekundärmetrik** | Recall Klasse 1 ≥ 0.50 |
| **Test-Split** | Chronologisch: Train 2016–2022 / Val 2023 / Test 2024 |
| **Hypothesen** | 7 Hypothesen, alle falsifizierbar, mit Feature-Zuordnung |
| **Hauptrisiko** | Klassenimbalance + fehlende Demografie-Features |

**Nächste Phase:** `02_U_Phase.ipynb` — EDA, Geo-Visualisierung, Feature Engineering.